In [27]:
import pandas as pd
import plotly.graph_objects as go

In [28]:
# Custom functions
from utils.visualisation_functions import plot_sankey_hierarchy
from utils.data_manipulations import build_activity_name, add_site_id
from core.lci_database_builder import LCIDatabaseBuilder
from utils.conversion_functions import map_technosphere_to_ecoinvent, map_biosphere_to_ecoinvent
from utils.constants import CA_provinces
from utils.data_manipulations import add_land_substance_name

# Plot selected sites

In [29]:
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [30]:
production_df_plot = production_df[production_df['Create_LCI?'] == 'Yes']
columns_to_plot = ['province', 'mining_processing_type', 'archetypes', 'Stream']

In [31]:
# plot_sankey_hierarchy(production_df_plot, columns_to_plot,
#                       html_output="data/MetalliCan/sites_for_lci_archetypes.html",
#                       output_image="data/MetalliCan/site_selection_sankey")

# Import cleaned MetalliCan data

In [32]:
# Pre-processed production table
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [33]:
# Add activitiy_name to production_df and site_id
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)
production_df = add_site_id(production_df)

In [34]:
# Normalized MetalliCan tables per ore processed
energy_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv')
material_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv')
biosphere_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv')
land_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/land_df.csv')
carbon_stock_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/carbon_stock_df.csv')

In [35]:
energy_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/energy_df.csv')
material_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/material_df.csv')
biosphere_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/biosphere_df.csv')
land_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/land_df.csv')
carbon_stock_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/carbon_stock_df.csv')

In [36]:
# Removing rows with value_normalized is NaN in the biosphere dfs
#biosphere_ore_df = biosphere_ore_df[~biosphere_ore_df['value_normalized'].isna()]
#biosphere_stream_df['value_normalized'] = biosphere_stream_df['value_normalized'] / 1e6

In [37]:
# Drop rows where 'unit' = 'ha'
biosphere_stream_df = biosphere_stream_df[biosphere_stream_df['unit'] != 'ha']
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_stream_df_bgf['unit'] != 'ha']

In [38]:
land_stream_df = add_land_substance_name(land_stream_df)
land_stream_df_bgf = add_land_substance_name(land_stream_df_bgf)

In [39]:
carbon_stock_stream_df.rename(columns={"flow_type": "substance_name"}, inplace=True)
carbon_stock_stream_df_bgf.rename(columns={"flow_type": "substance_name"}, inplace=True)

# Keeping only relevant columns

In [40]:
nrj_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
material_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
biosphere_col = ['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit', 'value_normalized']

In [41]:
energy_stream_df = energy_stream_df[nrj_col]
material_stream_df = material_stream_df[material_col]
biosphere_stream_df = biosphere_stream_df[biosphere_col]
land_stream_df = land_stream_df[biosphere_col]

In [42]:
energy_stream_df_bgf = energy_stream_df_bgf[nrj_col]
material_stream_df_bgf = material_stream_df_bgf[material_col]
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_col]
land_stream_df_bgf = land_stream_df_bgf[biosphere_col]

In [43]:
# Put energy_df and material_df together, and biosphere_df and land_df together
technosphere_stream_df = pd.concat([energy_stream_df, material_stream_df], ignore_index=True)
biosphere_stream_df = pd.concat([biosphere_stream_df, land_stream_df, carbon_stock_stream_df], ignore_index=True)
technosphere_stream_df_bgf = pd.concat([energy_stream_df_bgf, material_stream_df_bgf], ignore_index=True)
biosphere_stream_df_bgf = pd.concat([biosphere_stream_df_bgf, land_stream_df_bgf, carbon_stock_stream_df_bgf], ignore_index=True)

In [44]:
# Remove rows where value_normalized is NaN
technosphere_stream_df = technosphere_stream_df[~technosphere_stream_df['value_normalized'].isna()]
technosphere_stream_df_bgf = technosphere_stream_df_bgf[~technosphere_stream_df_bgf['value_normalized'].isna()]

In [45]:
# Add the province from the main_df to specify electricity location later
technosphere_stream_df = technosphere_stream_df.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
biosphere_stream_df = biosphere_stream_df.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
technosphere_stream_df_bgf = technosphere_stream_df_bgf.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
biosphere_stream_df_bgf = biosphere_stream_df_bgf.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')

# Map MetalliCan flows to EI and RI flows

## Technosphere flows

In [46]:
mapping_technosphere_ri = pd.read_excel(r'data/Mappings/MAPPINGS_RI.xlsx', sheet_name='technosphere')
mapping_technosphere_ei = pd.read_excel(r'data/Mappings/MAPPINGS_EI.xlsx', sheet_name='technosphere')

In [47]:
# Apply the function
mapped_technosphere_ri_df = map_technosphere_to_ecoinvent(technosphere_stream_df, mapping_technosphere_ri, CA_provinces)
mapped_technosphere_ei_df = map_technosphere_to_ecoinvent(technosphere_stream_df, mapping_technosphere_ei, CA_provinces)

⚠️ Les flux suivants n'ont pas trouvé de correspondance dans Ecoinvent:
 - Tailings|Other
⚠️ Les flux suivants n'ont pas trouvé de correspondance dans Ecoinvent:
 - Tailings|Other


In [48]:
mapped_technosphere_ri_df_bgf = map_technosphere_to_ecoinvent(technosphere_stream_df_bgf, mapping_technosphere_ri, CA_provinces)
mapped_technosphere_ei_df_bgf = map_technosphere_to_ecoinvent(technosphere_stream_df_bgf, mapping_technosphere_ei, CA_provinces)

In [49]:
# Drop rows where ecoinvent_flow_name is "No mapping" and Amount is NaN for now
mapped_technosphere_ri_df = mapped_technosphere_ri_df[(mapped_technosphere_ri_df["Activity"] != "No mapping")]
mapped_technosphere_ei_df = mapped_technosphere_ei_df[(mapped_technosphere_ei_df["Activity"] != "No mapping")]
mapped_technosphere_ri_df_bgf = mapped_technosphere_ri_df_bgf[(mapped_technosphere_ri_df_bgf["Activity"] != "No mapping")]
mapped_technosphere_ei_df_bgf = mapped_technosphere_ei_df_bgf[(mapped_technosphere_ei_df_bgf["Activity"] != "No mapping")]

In [50]:
technosphere_col_for_lci = ['site_id', 'activity_name', 'functional_unit', 'Amount', 'Amount_min', 'Amount_mean', 'Amount_max', 'Activity', 'Product', 'Unit', 'Location', 'Database']

In [51]:
mapped_technosphere_ri_df = mapped_technosphere_ri_df[technosphere_col_for_lci]
mapped_technosphere_ei_df = mapped_technosphere_ei_df[technosphere_col_for_lci]
mapped_technosphere_ri_df_bgf = mapped_technosphere_ri_df_bgf[technosphere_col_for_lci]
mapped_technosphere_ei_df_bgf = mapped_technosphere_ei_df_bgf[technosphere_col_for_lci]

## Biosphere flows mapping

In [52]:
mapping_biosphere_ri = pd.read_excel(r'data/Mappings/MAPPINGS_RI.xlsx', sheet_name='biosphere')
mapping_biosphere_ei = pd.read_excel(r'data/Mappings/MAPPINGS_EI.xlsx', sheet_name='biosphere')

In [53]:
mapped_biosphere_ri_df = map_biosphere_to_ecoinvent(biosphere_stream_df, mapping_biosphere_ri, CA_provinces)
mapped_biosphere_ei_df = map_biosphere_to_ecoinvent(biosphere_stream_df, mapping_biosphere_ei, CA_provinces)
mapped_biosphere_ri_df_bgf = map_biosphere_to_ecoinvent(biosphere_stream_df_bgf, mapping_biosphere_ri, CA_provinces)
mapped_biosphere_ei_df_bgf = map_biosphere_to_ecoinvent(biosphere_stream_df_bgf, mapping_biosphere_ei, CA_provinces)

Index(['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit',
       'value_normalized', 'normalization_key', 'allocation_factor',
       'mining_processing_type', 'archetypes', 'data_source',
       'reference_mass_unit', 'province', 'Type', 'substance_id',
       'compartment_name', 'release_pathway', 'flow_direction',
       'MetalliCan_unit', 'DB_to_map', 'Flow name', 'Compartments', 'Unit',
       'Comment', 'Alternatives'],
      dtype='object')
✅ Mapped 18863 biosphere flows (112 unique flows).
Index(['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit',
       'value_normalized', 'normalization_key', 'allocation_factor',
       'mining_processing_type', 'archetypes', 'data_source',
       'reference_mass_unit', 'province', 'Type', 'substance_id',
       'compartment_name', 'release_pathway', 'flow_direction',
       'MetalliCan_unit', 'DB_to_map', 'Flow name', 'Compartments', 'Unit',
       'Comment', 'Alternatives'],
      dtype='object')
✅

In [54]:
# Drop rows where ecoinvent_flow_name is "No mapping" and Amount is NaN for now
mapped_biosphere_ri_df = mapped_biosphere_ri_df[(mapped_biosphere_ri_df["Flow Name"] != "No mapping") & (~mapped_biosphere_ri_df["Amount"].isna())]
mapped_biosphere_ei_df = mapped_biosphere_ei_df[(mapped_biosphere_ei_df["Flow Name"] != "No mapping") & (~mapped_biosphere_ei_df["Amount"].isna())]
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf[(mapped_biosphere_ri_df_bgf["Flow Name"] != "No mapping") & (~mapped_biosphere_ri_df_bgf["Amount"].isna())]
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf[(mapped_biosphere_ei_df_bgf["Flow Name"] != "No mapping") & (~mapped_biosphere_ei_df_bgf["Amount"].isna())]

In [55]:
biosphere_col_for_lci = ['site_id', 'activity_name', 'functional_unit', 'Amount', 'Unit', 'Flow Name', 'Compartments', 'Database']

In [56]:
mapped_biosphere_ri_df = mapped_biosphere_ri_df[biosphere_col_for_lci]
mapped_biosphere_ei_df = mapped_biosphere_ei_df[biosphere_col_for_lci]
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf[biosphere_col_for_lci]
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf[biosphere_col_for_lci]

In [57]:
# Add province and commodities from NRCan to production table
mapped_biosphere_ri_df = mapped_biosphere_ri_df.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ei_df = mapped_biosphere_ei_df.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')

# LCI creation

## Regioinvent

In [64]:
# # Step 1 — initialize the builder
builder_regio = LCIDatabaseBuilder(db_name='metallican_lci_ri', project_name='metallican')

📂 Active Brightway project: metallican
🆕 Database 'metallican_lci_ri' created.


In [65]:
# # Step 2 — create the activity shells from the main dataframe
builder_regio.build_lci_entries(df=mapped_biosphere_ri_df)
print(len(builder_regio.lcis))

✅ Created 59 base LCI activities with production exchanges.
59


In [66]:
# # Step 3a — Populate with the technosphere exchanges
builder_regio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ri_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 218246 activities from Regioinvent
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff regionalized
✅ Added 808 technosphere exchanges.


In [67]:
# # Step 3b — Populate with the biosphere exchanges
builder_regio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ri_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
   ✅ Cached 110559 biosphere flows from biosphere3_spatialized_flows
✅ Added 13255 biosphere exchanges.


In [68]:
# # Step 4 - Consolidate duplicate flows
builder_regio.consolidate_exchanges()

🧮 Consolidation: 14122 → 1977 exchanges (summed duplicates).


In [69]:
builder_regio.write_to_database()

🧱 Writing 59 activities to database 'metallican_lci_ri'...
✅ Database 'metallican_lci_ri' processed successfully with 59 activities.


## Ecoinvent

In [70]:
# # Step 1 — initialize the builder
builder_ei = LCIDatabaseBuilder(db_name='metallican_lci_ei', project_name='metallican')

📂 Active Brightway project: metallican
🆕 Database 'metallican_lci_ei' created.


In [71]:
# # Step 2 — create the activity shells from the main dataframe
builder_ei.build_lci_entries(df=mapped_biosphere_ri_df)
print(len(builder_ei.lcis))

✅ Created 59 base LCI activities with production exchanges.
59


In [72]:
# # Step 3a — Populate with the technosphere exchanges
builder_ei.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 808 technosphere exchanges.


In [73]:
# # Step 3b — Populate with the biosphere exchanges
builder_ei.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 13255 biosphere exchanges.


In [74]:
# # Step 4 - Consolidate duplicate flows
builder_ei.consolidate_exchanges()

🧮 Consolidation: 14122 → 1940 exchanges (summed duplicates).


In [75]:
builder_ei.write_to_database()

🧱 Writing 59 activities to database 'metallican_lci_ei'...
✅ Database 'metallican_lci_ei' processed successfully with 59 activities.


# Before and after gap filling

### Before gap filling - technosphere only

In [76]:
# Step 1 — initialize the builder
builder_ei_bgf_tech = LCIDatabaseBuilder(db_name='metallican_bgf_tech', project_name='metallican')

📂 Active Brightway project: metallican
🆕 Database 'metallican_bgf_tech' created.


In [77]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_bgf_tech.build_lci_entries(df=mapped_biosphere_ei_df_bgf)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 58 base LCI activities with production exchanges.
58


In [78]:
# Step 3a — Populate with the technosphere exchanges
builder_ei_bgf_tech.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df_bgf)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 258 technosphere exchanges.


In [79]:
# Step 3b — Populate with the biosphere exchanges
#builder_ei_bgf_tech.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

In [80]:
# Step 4 - Consolidate duplicate flows
builder_ei_bgf_tech.consolidate_exchanges()

🧮 Consolidation: 316 → 244 exchanges (summed duplicates).


In [81]:
builder_ei_bgf_tech.write_to_database()

🧱 Writing 58 activities to database 'metallican_bgf_tech'...
✅ Database 'metallican_bgf_tech' processed successfully with 58 activities.


### Before gap filling - biosphere only

In [82]:
# Step 1 — initialize the builder
builder_ei_bgf_bio = LCIDatabaseBuilder(db_name='metallican_bgf_bio', project_name='metallican')

📂 Active Brightway project: metallican
🆕 Database 'metallican_bgf_bio' created.


In [83]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_bgf_bio.build_lci_entries(df=mapped_biosphere_ei_df_bgf)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 58 base LCI activities with production exchanges.
58


In [84]:
# Step 3a — Populate with the technosphere exchanges
#builder_ei_bgf_bio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

In [85]:
# Step 3b — Populate with the biosphere exchanges
builder_ei_bgf_bio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df_bgf)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 13193 biosphere exchanges.


In [86]:
# Step 4 - Consolidate duplicate flows
builder_ei_bgf_bio.consolidate_exchanges()

🧮 Consolidation: 13251 → 1292 exchanges (summed duplicates).


In [87]:
builder_ei_bgf_bio.write_to_database()

🧱 Writing 58 activities to database 'metallican_bgf_bio'...
✅ Database 'metallican_bgf_bio' processed successfully with 58 activities.


### After gap filling - technosphere only

In [88]:
# Step 1 — initialize the builder
builder_ei_agf_tech = LCIDatabaseBuilder(db_name='metallican_agf_tech', project_name='metallican')

📂 Active Brightway project: metallican
🆕 Database 'metallican_agf_tech' created.


In [89]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_agf_tech.build_lci_entries(df=mapped_biosphere_ei_df)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 59 base LCI activities with production exchanges.
58


In [90]:
# Step 3a — Populate with the technosphere exchanges
builder_ei_agf_tech.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 808 technosphere exchanges.


In [91]:
# Step 3b — Populate with the biosphere exchanges
#builder_ei_agf_tech.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

In [92]:
# Step 4 - Consolidate duplicate flows
builder_ei_agf_tech.consolidate_exchanges()

🧮 Consolidation: 867 → 672 exchanges (summed duplicates).


In [93]:
builder_ei_agf_tech.write_to_database()

🧱 Writing 59 activities to database 'metallican_agf_tech'...
✅ Database 'metallican_agf_tech' processed successfully with 59 activities.


### After gap filling - biosphere only

In [94]:
# Step 1 — initialize the builder
builder_ei_bgf_bio = LCIDatabaseBuilder(db_name='metallican_agf_bio', project_name='metallican')

📂 Active Brightway project: metallican
🆕 Database 'metallican_agf_bio' created.


In [95]:
# Step 2 — create the activity shells from the main dataframe
builder_ei_bgf_bio.build_lci_entries(df=mapped_biosphere_ei_df)
print(len(builder_ei_bgf_tech.lcis))

✅ Created 59 base LCI activities with production exchanges.
58


In [96]:
# Step 3a — Populate with the technosphere exchanges
#builder_ei_bgf_bio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

In [97]:
# Step 3b — Populate with the biosphere exchanges
builder_ei_bgf_bio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 13255 biosphere exchanges.


In [98]:
# Step 4 - Consolidate duplicate flows
builder_ei_bgf_bio.consolidate_exchanges()

🧮 Consolidation: 13314 → 1327 exchanges (summed duplicates).


In [99]:
builder_ei_bgf_bio.write_to_database()

🧱 Writing 59 activities to database 'metallican_agf_bio'...
✅ Database 'metallican_agf_bio' processed successfully with 59 activities.
